# TartanGround Dataset Downloader & Reformatter for Google Colab
This notebook automates downloading the TartanGround dataset using the official `tartanairpy` package and saving it directly to your mounted Google Drive in the format expected by the `bridgedepth` data loader.

## Key Optimizations & Design:
1. **Targeted Front View Only**: Configured to download only the **front camera view** (`lcam_front`, `rcam_front`) and **front disparity/depth** (`depth_lcam_front`). This saves massive amount of space (only ~15% of the full dataset size) and is fully supported by the robust dataloader.
2. **Parallel Downloading**: Utilizes `tartanairpy`'s multi-threaded downloader (`download_ground_multi_thread`) to download zip archives fast.
3. **Zero Local Storage Overhead**: Downloads are processed sequentially environment-by-environment. After each environment is extracted and copied to Google Drive, the local VM files are deleted and `drive.flush_and_unmount()` is called to flush and purge the DriveFS FUSE write cache (`/root/.config/Google/DriveFS`), reclaiming 100% of the local SSD space before starting the next environment.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# Install tartanairpy package directly from GitHub to get the latest downloader features
!pip install git+https://github.com/castacks/tartanairpy.git

In [ ]:
import os
import shutil
import time
import tartanair as ta

# ── Target Directory inside Google Drive ──────────────────────────────────
# The bridgedepth data loader expects: 
#  - datasets/TartanGround/
GDRIVE_TARTANGROUND_ROOT = "/content/drive/MyDrive/TartanGround"

# Sensor modalities needed by the dataloader
MODALITIES = ['image', 'depth']

# Robot platforms to download ('omni', 'diff', 'anymal'). 'omni' is standard.
VERSIONS = ['omni']

# Camera views required by bridgedepth (restricting to lcam_front and rcam_front only)
CAMERAS = [
    'lcam_front', 'rcam_front'
]

# Environments to process. You can edit this list to add more or disable environments.
# TartanGround contains 60+ environments. A subset is configured here by default.
ENVIRONMENTS = [
    {"name": "AbandonedCable", "enabled": True},
    {"name": "AbandonedFactory", "enabled": True},
    {"name": "AbandonedFactory2", "enabled": True},
    {"name": "AbandonedSchool", "enabled": True},
    {"name": "CarWelding", "enabled": True},
    {"name": "ModNeighborhood", "enabled": True},
    {"name": "OldtownSummer", "enabled": True},
    {"name": "NordicHarbor", "enabled": True},
    {"name": "ForestAutumn", "enabled": True},
    {"name": "ForestWinter", "enabled": True},
    {"name": "GreatMarsh", "enabled": True},
    {"name": "AmericanDiner", "enabled": True},
    {"name": "ArchVizTinyHouseDay", "enabled": True},
    {"name": "ArchVizTinyHouseNight", "enabled": True},
    {"name": "Antiquity3D", "enabled": True},
    {"name": "Apocalyptic", "enabled": True},
    {"name": "BrushifyMoon", "enabled": True},
]

In [ ]:
from google.colab import drive
import glob
import zipfile

if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(GDRIVE_TARTANGROUND_ROOT, exist_ok=True)

for idx, env_info in enumerate(ENVIRONMENTS):
    env_name = env_info["name"]
    if not env_info["enabled"]:
        print(f"\n>>> Skipping environment {idx+1}/{len(ENVIRONMENTS)}: {env_name}")
        continue
        
    print("\n" + "=" * 80)
    print(f"  PROCESSING ENVIRONMENT {idx+1}/{len(ENVIRONMENTS)}: {env_name}")
    print("=" * 80)
    
    # 0. Check and mount drive if unmounted
    if not os.path.exists("/content/drive/MyDrive"):
        print("[gdrive] Mount not active. Remounting Google Drive...")
        drive.mount('/content/drive')
        os.makedirs(GDRIVE_TARTANGROUND_ROOT, exist_ok=True)
        
    # Final destination path for this environment
    dest_env_path = os.path.join(GDRIVE_TARTANGROUND_ROOT, env_name)
    
    # Check if the environment folder already exists and is fully extracted to skip duplicate downloads
    if os.path.exists(dest_env_path) and len(glob.glob(os.path.join(dest_env_path, "**/*.png"), recursive=True)) > 0:
        print(f"[skip] Environment '{env_name}' already exists on Google Drive at {dest_env_path}. Skipping.")
        continue
        
    try:
        # 1. Initialize tartanair directly to the final Google Drive root path
        print(f"[tartanair] Initializing download directory directly on Google Drive: {GDRIVE_TARTANGROUND_ROOT}")
        ta.init(GDRIVE_TARTANGROUND_ROOT)
        
        # 2. Run multi-threaded download directly to Google Drive
        print(f"[tartanair] Launching multi-threaded download for '{env_name}'...")
        ta.download_ground_multi_thread(
            env=[env_name],
            version=VERSIONS,
            modality=MODALITIES,
            camera_name=CAMERAS,
            unzip=False,
            num_workers=8
        )
        
        # 3. Extract the downloaded zip files directly on Google Drive (recursive search for nested files)
        if not os.path.exists(dest_env_path):
            raise RuntimeError(f"Environment directory not found on Drive. Download might have failed for {env_name}.")
            
        zip_files = glob.glob(os.path.join(dest_env_path, "**/*.zip"), recursive=True)
        print(f"[extractor] Found {len(zip_files)} zip files on Drive under {dest_env_path}. Extracting in-place...")
        
        for zip_file in sorted(zip_files):
            parent_dir = os.path.dirname(zip_file)
            zip_name = os.path.basename(zip_file)
            folder_name = zip_name.replace('.zip', '')
            
            # Verify if the zip file contains the folder inside or if we need to create it
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                namelist = zip_ref.namelist()
                files = [x for x in namelist if not x.endswith('/')]
                if len(files) == 0:
                    extract_dest = os.path.join(parent_dir, folder_name)
                else:
                    first_file = files[0]
                    if '/' in first_file:
                        extract_dest = parent_dir
                    else:
                        extract_dest = os.path.join(parent_dir, folder_name)
                    
            print(f"[extractor] Extracting {zip_name} to {extract_dest}...")
            os.makedirs(extract_dest, exist_ok=True)
            cmd = f"unzip -q -o '{zip_file}' -d '{extract_dest}'"
            status = os.system(cmd)
            if status != 0:
                print(f"[WARNING] Extraction command returned non-zero code for {zip_file}")
            else:
                # Delete zip file after successful extraction to save space on Google Drive
                os.remove(zip_file)
                
        print(f"[success] Successfully downloaded and extracted '{env_name}' to final destination!")
            
    except Exception as e:
        print(f"[ERROR] Processing failed for environment '{env_name}': {e}")
        print("Stopping pipeline. Please check errors and resume.")
        break

# Final check: ensure drive is mounted
if not os.path.exists("/content/drive/MyDrive"):
    print("\n[gdrive] Remounting Google Drive...")
    drive.mount('/content/drive')
        
print("\n" + "=" * 80)
print("  PIPELINE PROCESSING COMPLETED")
print("=" * 80)
